# Validation 1 figures: pair selection and PyMOL scripts

Two-step workflow:

1. **Browse candidate pairs interactively** with py3Dmol (this notebook). Rotate with your mouse to see the three structures (crystal/decoy/template) in 3D and decide which pairs you want to feature.
2. **Save a `.pml` script for each chosen pair**. Open the `.pml` in PyMOL GUI, orient by hand, then ray-trace.

**Color convention** (used in both viewer and PyMOL):
- Blue — experimental crystal peptide
- Orange — regenerated lowest-score decoy
- Grey (lines) — threading template peptide (context only)
- Pale grey, semi-transparent cartoon + surface — crystal MHC binding cleft

## Setup

In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image as IPImage, display

# Make the validation utils package importable
VALIDATION_DIR = Path('/home/huntek1/main_project/scripts/IEDB_validation')
sys.path.insert(0, str(VALIDATION_DIR))
from utils import allele_to_dir_name, get_decoy_paths

In [2]:
# === Paths ===
VALIDATION_OUT = Path('/home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/01_structural_regen')
PDB_ROOT       = Path('/home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/regeneration/pdb')
TEMPLATE_DIR   = Path('/home/huntek1/Data/MHC_database/templates')
FIG_OUT = Path('/home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/figure2_panels')
FIG_OUT.mkdir(parents=True, exist_ok=True)

COL_ALLELE  = 'allele'
COL_PEPTIDE = 'peptide'

# Verify pymol binary is available
PYMOL_BIN = shutil.which('pymol') or shutil.which('pymol3')
print(f'PyMOL binary: {PYMOL_BIN}')
if not PYMOL_BIN:
    print('WARNING: pymol not in PATH. Activate your pymol conda env before launching Jupyter,')
    print("or 'conda activate <env>' in a terminal and start Jupyter from there.")

PyMOL binary: /home/huntek1/miniconda3/envs/pymol-env/bin/pymol


## Load validation results

In [3]:
rmsd_df = pd.read_csv(VALIDATION_OUT / 'rmsd_per_pair.csv')
if 'template_identity' not in rmsd_df.columns:
    raise RuntimeError(
        'template_identity column missing from rmsd_per_pair.csv. '
        'Rerun notebook 01 through the template-similarity section, '
        'then re-save the CSV with: rmsd_df.to_csv(OUT_DIR / "rmsd_per_pair.csv", index=False)'
    )
print(f'Loaded {len(rmsd_df)} validation pairs')

Loaded 52 validation pairs


## Candidate menu by tier

Use this to pick which pair to visualize per tier. Inspect the candidates in the interactive viewer (next section) before deciding.

In [4]:
print('HIGH identity (>=80%), best RMSD first:')
print(rmsd_df[rmsd_df['template_identity'] >= 80][
    [COL_ALLELE, COL_PEPTIDE, 'template_identity', 'rmsd_best_score', 'matched_pdb_id', 'resolution_angstrom']
].sort_values('rmsd_best_score').head(8).to_string(index=False))

print('\nLOW identity (<=35%) with RMSD < 1.5 A (the punchline tier):')
print(rmsd_df[(rmsd_df['template_identity'] <= 35) & (rmsd_df['rmsd_best_score'] <= 1.5)][
    [COL_ALLELE, COL_PEPTIDE, 'template_identity', 'rmsd_best_score', 'matched_pdb_id', 'resolution_angstrom']
].sort_values('rmsd_best_score').head(8).to_string(index=False))

print('\nMODERATE (35-60%) identity, RMSD near median:')
print(rmsd_df[(rmsd_df['template_identity'] > 35) & (rmsd_df['template_identity'] < 60)][
    [COL_ALLELE, COL_PEPTIDE, 'template_identity', 'rmsd_best_score', 'matched_pdb_id', 'resolution_angstrom']
].sort_values('rmsd_best_score').head(8).to_string(index=False))

print('\nOUTLIERS (RMSD > 2 A):')
print(rmsd_df[rmsd_df['rmsd_best_score'] > 2.0][
    [COL_ALLELE, COL_PEPTIDE, 'peptide_length', 'template_identity', 'rmsd_best_score', 'matched_pdb_id']
].sort_values('rmsd_best_score', ascending=False).to_string(index=False))

HIGH identity (>=80%), best RMSD first:
 allele    peptide  template_identity  rmsd_best_score matched_pdb_id  resolution_angstrom
B*57:01  LSSPVTKSF          88.888889         0.352090           3VH8                 1.80
A*02:01  ALWGFFPVL          88.888889         0.466460           1LP9                 2.00
A*01:01  CTELKLSDY          88.888889         0.635936           4NQV                 2.39
B*35:01  LPFDKSTIM          88.888889         0.678161           3LKP                 1.80
A*02:01  ITDQVPFSV          88.888889         0.750394           1TVB                 1.80
A*02:01 ALWGPDPAAA          90.000000         0.785035           3UTQ                 1.67
A*02:01  VLHDDLLEA          88.888889         0.890997           3D25                 1.30
A*02:01  RLQSLQTYV          88.888889         0.903757           7N1E                 2.30

LOW identity (<=35%) with RMSD < 1.5 A (the punchline tier):
 allele    peptide  template_identity  rmsd_best_score matched_pdb_id  resoluti

## Recover threading templates

In [5]:
from Bio.Align import substitution_matrices
from Bio import pairwise2

mhc_db = pd.read_csv('/home/huntek1/Data/MHC_database/database.info')
matrix = substitution_matrices.load('BLOSUM62')

def best_template_for_pair(allele, peptide):
    candidates = mhc_db[
        (mhc_db['MHC_Allele'] == allele) &
        (mhc_db['Epitope_Description'].str.len() == len(peptide))
    ].copy()
    candidates = candidates[candidates['Epitope_Description'] != peptide]
    if len(candidates) == 0:
        return None
    best_score = -float('inf')
    best_row = None
    for _, row in candidates.iterrows():
        alignments = pairwise2.align.globalds(
            peptide, row['Epitope_Description'], matrix, -10, -0.5
        )
        score = max(a.score for a in alignments)
        if score > best_score:
            best_score = score
            best_row = row
    return best_row


def get_pair_info(allele, peptide):
    row = rmsd_df[(rmsd_df[COL_ALLELE] == allele) & (rmsd_df[COL_PEPTIDE] == peptide)]
    if len(row) == 0:
        raise ValueError(f'Pair {allele}/{peptide} not in rmsd_df')
    row = row.iloc[0]

    t = best_template_for_pair(allele, peptide)
    if t is None:
        raise ValueError(f'No non-self template available for {allele}/{peptide}')

    allele_dir = allele_to_dir_name(allele)
    _, decoy_paths, score_df = get_decoy_paths(PDB_ROOT, allele_dir, peptide)
    best_idx = int(np.argmin(score_df['total_score'].values))

    return {
        'allele':              allele,
        'peptide':             peptide,
        'crystal_pdb':         row['matched_pdb_id'],
        'crystal_path':        str(TEMPLATE_DIR / f'{row["matched_pdb_id"].upper()}.pdb'),
        'crystal_mhc_chain':   str(row['mhc_chain_id']).strip(),
        'crystal_pep_chain':   str(row['peptide_chain_id']).split(',')[0].strip(),
        'decoy_path':          str(decoy_paths[best_idx]),
        'template_pdb':        t['PDB_ID'],
        'template_path':       str(TEMPLATE_DIR / f'{t["PDB_ID"].upper()}.pdb'),
        'template_peptide':    t['Epitope_Description'],
        'template_mhc_chain':  str(t['MHC_PDB_Chain1']).strip(),
        'template_pep_chain':  str(t['Antigen_PDB_Chain(s)']).split(',')[0].strip(),
        'template_identity':   row['template_identity'],
        'rmsd_best_score':     row['rmsd_best_score'],
        'rmsd_min_of_25':      row['rmsd_min_of_25'],
    }


def print_pair_summary(info):
    print(f"{info['allele']} {info['peptide']}")
    print(f"  Crystal:  {info['crystal_pdb']}")
    print(f"  Template: {info['template_pdb']} (peptide {info['template_peptide']}, "
          f"{info['template_identity']:.1f}% identity)")
    print(f"  RMSD: best {info['rmsd_best_score']:.2f} A, "
          f"min-of-25 {info['rmsd_min_of_25']:.2f} A")

/home/huntek1/.local/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


## Interactive viewer (browse candidates)

Use `interactive_view(get_pair_info(allele, peptide))` to inspect any pair. Mouse: drag to rotate, scroll to zoom, shift+drag to pan.

In [6]:
import py3Dmol
import shutil
import subprocess


# Cache for aligned PDBs (one folder per pair)
ALIGNED_CACHE = FIG_OUT / '_aligned_cache'
ALIGNED_CACHE.mkdir(parents=True, exist_ok=True)


def get_aligned_pdbs(info):
    """Run PyMOL once to align decoy + template MHC chains onto crystal MHC.
    Save aligned PDBs in a per-pair cache directory. Returns the three paths."""
    pair_key = f"{info['allele'].replace('*','').replace(':','')}_{info['peptide']}"
    pair_dir = ALIGNED_CACHE / pair_key
    pair_dir.mkdir(parents=True, exist_ok=True)

    crystal_out  = pair_dir / 'crystal.pdb'
    decoy_out    = pair_dir / 'decoy_aligned.pdb'
    template_out = pair_dir / 'template_aligned.pdb'

    # Reuse if already aligned
    if all(p.exists() for p in (crystal_out, decoy_out, template_out)):
        return crystal_out, decoy_out, template_out

    # Run PyMOL to do the alignment
    script = f"""
load {info['crystal_path']}, crystal
load {info['decoy_path']}, decoy
load {info['template_path']}, template
cealign crystal and chain {info['crystal_mhc_chain']}, decoy and chain A
cealign crystal and chain {info['crystal_mhc_chain']}, template and chain {info['template_mhc_chain']}
delete *aln*
save {crystal_out}, crystal
save {decoy_out}, decoy
save {template_out}, template
quit
"""
    align_pml = pair_dir / '_align.pml'
    align_pml.write_text(script)

    pymol_bin = shutil.which('pymol') or shutil.which('pymol3')
    if pymol_bin is None:
        raise RuntimeError('pymol not in PATH - activate your pymol-env')

    result = subprocess.run([pymol_bin, '-cq', str(align_pml)],
                            capture_output=True, text=True, timeout=120)
    if not all(p.exists() for p in (crystal_out, decoy_out, template_out)):
        print('PyMOL alignment failed; stderr:')
        print(result.stderr[-600:])
        raise RuntimeError('alignment failed')

    return crystal_out, decoy_out, template_out


def interactive_view(info, width=650, height=500):
    """Render three structures (pre-aligned by PyMOL) with mouse rotation.
    Use this to decide whether a pair makes a good figure example."""
    crystal_path, decoy_path, template_path = get_aligned_pdbs(info)

    v = py3Dmol.view(width=width, height=height)
    v.addModel(open(crystal_path).read(),  'pdb')
    v.addModel(open(decoy_path).read(),    'pdb')
    v.addModel(open(template_path).read(), 'pdb')

    v.setStyle({'model': 0, 'chain': info['crystal_mhc_chain'], 'resi': '1-180'},
               {'cartoon': {'color': 'lightgray', 'opacity': 0.5}})
    v.setStyle({'model': 0, 'chain': info['crystal_pep_chain']},
               {'stick': {'color': 'forestgreen', 'radius': 0.2}})
    v.setStyle({'model': 1, 'chain': 'B'},
               {'stick': {'color': 'royalblue', 'radius': 0.2}})
    v.setStyle({'model': 2, 'chain': info['template_pep_chain']},
               {'stick': {'color': 'firebrick', 'radius': 0.2}})

    v.zoomTo({'model': 0, 'chain': info['crystal_pep_chain']})
    v.show()
    # Note: no return — prevents Jupyter from auto-displaying a second viewer

### Inspect a candidate

Change the `(allele, peptide)` to look at different pairs. Repeat this cell as needed by editing and rerunning.

In [7]:
info = get_pair_info('A*02:01', 'GMSRIGMEV')   # ← edit to inspect different pairs
print_pair_summary(info)
interactive_view(info)

A*02:01 GMSRIGMEV
  Crystal:  7KGP
  Template: 6R2L (peptide SLSKILDTV, 33.3% identity)
  RMSD: best 1.13 A, min-of-25 1.05 A


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Save `.pml` scripts for the chosen pairs

Once you've decided which four pairs (or however many) you want to feature, run the cells below to generate one `.pml` per pair. Each script:

- Loads the three structures
- Aligns the decoy and template MHC chains onto the crystal MHC
- Hides scaffolding (the aligned-but-unused MHC chains)
- Sets up the color scheme and transparency
- Does **not** render — leaves the session interactive in PyMOL GUI for you to orient and ray-trace by hand

**Opening in PyMOL GUI:**

```bash
cd /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/01_structural_regen/pymol_figures
pymol panel_A_GILGFVFTL.pml
```

Or from the PyMOL command bar inside an already-open session:

```
@panel_A_GILGFVFTL.pml
```

**To render the final image after orienting:**

```
ray 1600, 1200
png panel_A_GILGFVFTL.png, dpi=300
```

In [10]:
PYMOL_SETUP_BLOCK = """# --- Visual settings ---
bg_color white
set ray_trace_mode, 1
set ray_trace_color, black
set ray_opaque_background, 1
set ray_trace_gain, 0.01
set ray_shadows, off
set antialias, 2
set spec_reflect, 0
set spec_power, 0
set ambient, 0.5
set depth_cue, 0
set two_sided_lighting, on
set cartoon_fancy_helices, 1
set stick_quality, 25
"""

def write_pml_for_pair(info, panel_tag, out_dir=None, mhc_transparency=0.3,
                       template_transparency=0.4):
    """Write a .pml that sets up the scene but does NOT ray-trace.
    User opens it in PyMOL GUI, orients with the mouse, then renders.
    Returns the path to the .pml file.
    """
    if out_dir is None:
        out_dir = FIG_OUT
    pml_path = out_dir / f"{panel_tag}_{info['peptide']}.pml"

    crystal_pep_chain  = info['crystal_pep_chain']
    crystal_mhc_chain  = info['crystal_mhc_chain']
    template_pep_chain = info['template_pep_chain']
    template_mhc_chain = info['template_mhc_chain']

    script = f"""{PYMOL_SETUP_BLOCK}
# --- Pair info ---
# Panel:           {panel_tag}
# Allele:          {info['allele']}
# Peptide:         {info['peptide']} ({len(info['peptide'])}-mer)
# Crystal PDB:     {info['crystal_pdb']}
# Threading PDB:   {info['template_pdb']} (peptide {info['template_peptide']})
# Template identity: {info['template_identity']:.1f}%
# Validation: best-score RMSD = {info['rmsd_best_score']:.2f} A
#             min-of-25 RMSD = {info['rmsd_min_of_25']:.2f} A

# --- Load structures ---
load {info['crystal_path']},  crystal
load {info['decoy_path']},    decoy
load {info['template_path']}, template

# --- Align decoy and template MHC chains onto crystal MHC ---
# (cealign auto-shows things; we hide everything after this step)
cealign crystal and chain {crystal_mhc_chain}, decoy and chain A
cealign crystal and chain {crystal_mhc_chain}, template and chain {template_mhc_chain}

python
for obj in cmd.get_object_list():
    if obj not in ("crystal", "decoy", "template"):
        cmd.delete(obj)
python end

# --- Hide everything, then selectively show what we want ---
hide everything

# Crystal MHC binding cleft: cartoon + mesh wireframe for shape context
show cartoon, crystal and chain {crystal_mhc_chain} and resi 1-180
color grey90, crystal and chain {crystal_mhc_chain} and resi 1-180
#set cartoon_transparency, 0.2, crystal and chain {crystal_mhc_chain} and resi 1-180

# Crystal peptide: blue sticks (full atom)
show sticks, crystal and chain {crystal_pep_chain}
color tv_blue, crystal and chain {crystal_pep_chain}
set stick_radius, 0.22, crystal and chain {crystal_pep_chain}

# Decoy peptide: orange sticks (modeled structures have peptide in chain B)
show sticks, decoy and chain B
color tv_orange, decoy and chain B
set stick_radius, 0.22, decoy and chain B

# Template peptide: thin cartoon ribbon (de-emphasized context)
show cartoon, template and chain {template_pep_chain}
color grey50, template and chain {template_pep_chain}
set cartoon_tube_radius, 0.1, template and chain {template_pep_chain}
set cartoon_smooth_loops, 1, template and chain {template_pep_chain}

# --- Suggested starting view (you'll likely want to rotate further) ---
orient crystal and chain {crystal_pep_chain}
zoom crystal and chain {crystal_pep_chain}, 6

# --- DO NOT add ray/png/quit here ---
# The session stays interactive in PyMOL GUI for manual orientation.
# Once oriented, render with:
#     ray 1600, 1200
#     png {panel_tag}_{info['peptide']}.png, dpi=300
"""
    with open(pml_path, 'w') as f:
        f.write(script)
    return pml_path

### Generate scripts for the chosen pairs

Edit the `PAIRS_TO_RENDER` list with whichever pairs you decided on after browsing. Each entry is `(panel_tag, allele, peptide)`.

In [11]:
# Define the tiers we want to bucket pairs into
def assign_tier(row):
    """Return tier name based on identity and RMSD, or None if pair doesn't fit."""
    ident = row.get('template_identity')
    rmsd  = row['rmsd_best_score']

    if pd.isna(ident):
        return 'no_template'             # the 2 pairs with no non-self template
    if rmsd > 2.0:
        return 'outliers'
    if ident >= 80:
        return 'high_identity'
    if ident <= 35:
        return 'low_identity'
    return 'moderate_identity'


# Bucket every pair (skip the 2 with no template — they have no template to render)
all_pairs = []
for _, row in rmsd_df.iterrows():
    tier = assign_tier(row)
    if tier == 'no_template':
        continue
    all_pairs.append((tier, row[COL_ALLELE], row[COL_PEPTIDE]))

# Group counts so you know what you're getting
from collections import Counter
tier_counts = Counter(t for t, _, _ in all_pairs)
print('Pairs per tier:')
for tier, count in sorted(tier_counts.items()):
    print(f'  {tier:20s} {count:3d}')
print(f'  {"total":20s} {len(all_pairs):3d}')
print()


# Write one .pml per pair, organized into tier subdirectories
written = []
skipped = []
for tier, allele, peptide in all_pairs:
    try:
        info = get_pair_info(allele, peptide)
    except ValueError as e:
        skipped.append((tier, allele, peptide, str(e)))
        continue

    tier_dir = FIG_OUT / tier
    tier_dir.mkdir(parents=True, exist_ok=True)

    # Use the standard write_pml_for_pair, but redirect output to the tier subdir
    # Simplest way: monkey-patch FIG_OUT temporarily — or just inline a small wrapper.
    panel_tag = f'{tier}_{allele.replace("*","").replace(":","")}'
    path = write_pml_for_pair(info, panel_tag, out_dir=tier_dir)
    written.append(path)

print(f'Wrote {len(written)} .pml scripts across {len(tier_counts)} tier directories:')
for tier in sorted(tier_counts):
    print(f'  {FIG_OUT / tier}/')

if skipped:
    print(f'\nSkipped {len(skipped)} pairs:')
    for tier, allele, peptide, reason in skipped:
        print(f'  {tier} / {allele} / {peptide}: {reason}')

Pairs per tier:
  high_identity         17
  low_identity          16
  moderate_identity     10
  outliers               7
  total                 50

Wrote 50 .pml scripts across 4 tier directories:
  /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/figure2_panels/high_identity/
  /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/figure2_panels/low_identity/
  /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/figure2_panels/moderate_identity/
  /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/figure2_panels/outliers/


## Rendering in PyMOL GUI — quick reference

From a terminal on `tungsten` (with X11 forwarding enabled in your SSH session, or if you've copied the files to your local machine):

```bash
cd /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/01_structural_regen/pymol_figures
pymol panel_A_high_identity_GILGFVFTL.pml
```

Once the structures load:

1. Drag with **left mouse button** to rotate.
2. **Scroll wheel** to zoom in/out.
3. **Right mouse button** drag to translate.
4. Middle-click on an atom to set the rotation center.

Once you have the view you want, in the PyMOL command bar at the top:

```
ray 800, 600
png panel_A_high_identity_GILGFVFTL.png, dpi=300
```

The PNG saves to wherever PyMOL's working directory is — usually the same folder the `.pml` was loaded from.

**Customizing the visuals while in PyMOL:**
- Increase MHC transparency: `set cartoon_transparency, 0.7, crystal`
- Hide the MHC entirely: `hide cartoon, crystal`
- Make the template more faded: `set stick_transparency, 0.6, template`
- Show the H-bonds between peptide and MHC: `distance hb, crystal and chain {pep_chain}, crystal and chain {mhc_chain}, 3.5, mode=2`

**For X11 forwarding from VS Code over SSH** (if you don't already have it set up):
in your SSH config or with `ssh -Y huntek1@tungsten...`, then `xclock` to test X11 is working before launching PyMOL. If you're working on your local machine directly, you don't need X11 — just install PyMOL locally.

## Combined render for figure panels A/B/C

In [13]:
from pathlib import Path

# ===== Picks =====
PICKS = [
    ('a', 'A*01:01', 'EADPTGHSY'),
    ('b', 'B*15:01', 'VQQESSFVM'),
    ('c', 'B*07:02', 'RPPIFIRRL'),
]

OUT_DIR = FIG_OUT / 'combined'
OUT_DIR.mkdir(exist_ok=True)
COMBINED_PML = OUT_DIR / 'figure2_combined.pml'

# Look up info for each pick using the existing get_pair_info function
infos = {p: get_pair_info(allele, peptide) for p, allele, peptide in PICKS}

# Same setup block as your per-pair .pml files
SETUP = """# --- Visual settings ---
bg_color white
set ray_trace_mode, 1
set ray_trace_color, black
set ray_opaque_background, 1
set ray_trace_gain, 0.01
set ray_shadows, off
set antialias, 2
set spec_reflect, 0
set spec_power, 0
set ambient, 0.5
set depth_cue, 0
set two_sided_lighting, on
set cartoon_fancy_helices, 1
set stick_quality, 25
"""

def load_block(p, info):
    return f"""
# === Panel {p.upper()}: {info['allele']} / {info['peptide']} ===
# Crystal: {info['crystal_pdb']} | Template: {info['template_pdb']} ({info['template_identity']:.1f}% ID) | RMSD: {info['rmsd_best_score']:.2f} A
load {info['crystal_path']},  {p}_crystal
load {info['decoy_path']},    {p}_decoy
load {info['template_path']}, {p}_template

# Intra-pair alignment: decoy and template MHCs onto this pair's crystal MHC
cealign {p}_crystal and chain {info['crystal_mhc_chain']}, {p}_decoy and chain A
cealign {p}_crystal and chain {info['crystal_mhc_chain']}, {p}_template and chain {info['template_mhc_chain']}
"""

def cross_pair_align(infos):
    a = infos['a']
    return f"""
# === Cross-pair alignment: bring B and C into A's coordinate frame ===
cealign a_crystal  and chain {a['crystal_mhc_chain']}, b_crystal  and chain {infos['b']['crystal_mhc_chain']}
cealign a_crystal  and chain {a['crystal_mhc_chain']}, b_decoy    and chain A
cealign a_crystal  and chain {a['crystal_mhc_chain']}, b_template and chain {infos['b']['template_mhc_chain']}
cealign a_crystal  and chain {a['crystal_mhc_chain']}, c_crystal  and chain {infos['c']['crystal_mhc_chain']}
cealign a_crystal  and chain {a['crystal_mhc_chain']}, c_decoy    and chain A
cealign a_crystal  and chain {a['crystal_mhc_chain']}, c_template and chain {infos['c']['template_mhc_chain']}
"""

CLEANUP = """
# Delete any extra objects that cealign may have created
python
keep = {"a_crystal","a_decoy","a_template","b_crystal","b_decoy","b_template","c_crystal","c_decoy","c_template"}
for obj in cmd.get_object_list():
    if obj not in keep:
        cmd.delete(obj)
python end
"""

def style_block(p, info):
    return f"""
# === Style Panel {p.upper()} ===
# Crystal MHC binding cleft: grey90 cartoon
show cartoon, {p}_crystal and chain {info['crystal_mhc_chain']} and resi 1-180
color grey90, {p}_crystal and chain {info['crystal_mhc_chain']} and resi 1-180

# Crystal peptide: blue sticks
show sticks, {p}_crystal and chain {info['crystal_pep_chain']}
color tv_blue, {p}_crystal and chain {info['crystal_pep_chain']}
set stick_radius, 0.22, {p}_crystal and chain {info['crystal_pep_chain']}

# Decoy peptide: orange sticks (decoys always have peptide in chain B)
show sticks, {p}_decoy and chain B
color tv_orange, {p}_decoy and chain B
set stick_radius, 0.22, {p}_decoy and chain B

# Template peptide: thin grey cartoon ribbon
show cartoon, {p}_template and chain {info['template_pep_chain']}
color grey50, {p}_template and chain {info['template_pep_chain']}
set cartoon_tube_radius, 0.1, {p}_template and chain {info['template_pep_chain']}
set cartoon_smooth_loops, 1, {p}_template and chain {info['template_pep_chain']}
"""

HELPERS = f"""
# === Python helpers ===
python
import __main__
from pymol import cmd

OUT_DIR = "{OUT_DIR}"

def _show_only(prefix):
    cmd.disable('*')
    for kind in ('crystal', 'decoy', 'template'):
        cmd.enable(f'{{prefix}}_{{kind}}')

def show_a(): _show_only('a')
def show_b(): _show_only('b')
def show_c(): _show_only('c')

def render_panel(prefix):
    _show_only(prefix)
    path = f'{{OUT_DIR}}/panel_{{prefix.upper()}}.png'
    cmd.ray(2400, 1800)
    cmd.png(path, dpi=300)
    print(f'Saved {{path}}')

def render_all():
    view = cmd.get_view()
    for p in ('a','b','c'):
        cmd.set_view(view)
        render_panel(p)

for name in ('show_a','show_b','show_c','render_panel','render_all'):
    fn = locals()[name]
    setattr(__main__, name, fn)
    cmd.extend(name, fn)
python end
"""

# Assemble
parts = [SETUP]
for p, _, _ in PICKS:
    parts.append(load_block(p, infos[p]))
parts.append(cross_pair_align(infos))
parts.append(CLEANUP)
parts.append("\n# Hide everything (no selection = all objects), then selectively style")
parts.append("hide everything")
for p, _, _ in PICKS:
    parts.append(style_block(p, infos[p]))
parts.append(HELPERS)
parts.append("\n# Default: show Panel A and orient on its peptide")
parts.append("show_a")
parts.append(f"orient a_crystal and chain {infos['a']['crystal_pep_chain']}")
parts.append(f"zoom a_crystal and chain {infos['a']['crystal_pep_chain']}, 6")

COMBINED_PML.write_text('\n'.join(parts) + '\n')

print(f"Wrote {COMBINED_PML}")
print()
print("Next steps:")
print(f"  pymol {COMBINED_PML}")
print( "  # orient on Panel A with the mouse, then:")
print( "  render_all")

Wrote /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/figure2_panels/combined/figure2_combined.pml

Next steps:
  pymol /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/figure2_panels/combined/figure2_combined.pml
  # orient on Panel A with the mouse, then:
  render_all
